In [0]:
df_bronze_patients = spark.read.table("workspace.bronze.patients")
df_bronze_patients.printSchema()
df_bronze_patients.show(20, truncate=False)

In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, trim
import pyspark.sql.functions as F

df_patients_bronze = spark.read.table("workspace.bronze.patients")

df_patients_silver = (
    df_patients_bronze
    .filter(col("patient").isNotNull())
    .dropDuplicates(["patient"])
    .withColumn("patient_name", trim(F.concat_ws(" ", col("first"), col("last"))))
    # Extract state code (e.g., 'MA' from 'Pittsfield MA US') using F.element_at & F.split
    .withColumn("state", trim(F.element_at(F.split(col("birthplace"), r"\s+"), -2)))
    .select(
        col("patient").alias("patient_id"),
        col("patient_name"),
        col("birthdate"),
        col("deathdate"),
        col("ssn"),
        col("drivers").alias("drivers_license"),
        col("passport"),
        trim(col("birthplace")).alias("birthplace"),
        col("state"),
        F.upper(trim(col("gender"))).alias("gender"),
        trim(col("ethnicity")).alias("ethnicity"),
        trim(col("race")).alias("race"),
        trim(col("marital")).alias("marital_status"),
        trim(col("address")).alias("address"),
        col("ingested_at")
    )
)

(
    df_patients_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.patients")
)

print(f"✅ Created workspace.silver.patients with {df_patients_silver.count()} clean rows!")